# 02 · Uncertainty fingerprints (≈ 12 min)
**Challenge 3: build a fingerprint and find the confident fabrication the signals miss.**

A **fingerprint** is the cell of the (SC, SE) plane an answer falls in. We cut each axis at its **median**:

| Region | SC | SE | Reading |
|---|---|---|---|
| Confident | low | low | samples agree |
| Gap-like | high | high | samples disagree |
| Shallow | low | high | strings agree, meanings differ |
| Surface-varying | high | low | wording differs, meaning agrees |

No labels are used to build it, so it could run in a real system.

In [ ]:
import sys, pathlib, warnings
from collections import Counter
REPO = "pycon2026-uncertainty-workshop"   # <- must match your GitHub repo name (used on Colab only)
if "google.colab" in sys.modules:
    ROOT = pathlib.Path("/content") / REPO
    if not ROOT.exists():
        import subprocess
        subprocess.run(["git", "clone", f"https://github.com/Shafiya0101/{REPO}.git", str(ROOT)], check=True)
else:
    ROOT = pathlib.Path.cwd()
    if ROOT.name in ("notebooks", "solutions"):
        ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from uncertainty_lib import *
from uncertainty_lib import checks
from uncertainty_lib.plotting import set_style, plot_region_grid, plot_risk_coverage
set_style()
pd.set_option("display.max_colwidth", 70)

In [ ]:
df = add_signals(load_setting("mistral_triviaqa"))
checks.banner(df)
from scipy.stats import spearmanr
rho = spearmanr(df.U_sc, df.U_se).statistic
print(f"Rank correlation SC vs SE: {rho:.2f}   (close to 1 means they carry almost the same information)")
tau_sc, tau_se = fit_medians(df)
print(f"Median thresholds: tau_SC = {tau_sc:.2f}, tau_SE = {tau_se:.2f}")

### 🛠 TODO 3: write the region rule

In [ ]:
def my_region(sc, se, tau_sc, tau_se):
    sc_high, se_high = sc > tau_sc, se > tau_se
    if not sc_high and not se_high: return "Confident"
    if sc_high and se_high:         return "Gap-like"
    if not sc_high and se_high:     return "Shallow"
    return "Surface-varying"

In [ ]:
checks.check_region(my_region, tau_sc, tau_se)

In [ ]:
df["region"] = [my_region(a, b, tau_sc, tau_se) for a, b in zip(df.U_sc, df.U_se)]
summary = region_summary(df)
print(summary.round(3))
plot_region_grid(summary)
plt.show()

**Observe:** how many regions have enough questions to say anything? Usually only *Confident* and *Gap-like* are well populated.
The error rate in *Gap-like* is the highest. That is the information that actually matters.

## Challenge 3 · The confident fabrication
"Confident" by the median is only *relative*. Use an **absolute** definition: all 10 samples identical (`U_SC = 0`) **and** the answer is wrong.

In [ ]:
bad = unanimous_errors(df)
print(f"{len(bad)} unanimous-but-wrong answers out of {len(df)} questions")
pct = lambda c: df[c].rank(pct=True)          # 0 = most certain, 1 = most uncertain
view = bad.assign(LP_pct=pct("U_lp")[bad.index], NLI_pct=pct("U_nli")[bad.index],
                  Comb4_pct=pct("Comb4")[bad.index])
view[["question", "greedy", "gold", "region", "LP_pct", "NLI_pct", "Comb4_pct"]].round(2)

**Questions to answer in pairs (3 min)**
1. Which region do these answers fall in? Could a policy that abstains on *Gap-like* ever catch them?
2. Do LP or NLI flag them? (a percentile near 1.0 = flagged, near 0 = invisible)
3. What kind of evidence *outside the model* could catch them?

**Bonus (fast finishers):** change `"mistral_triviaqa"` to `"qwen2_1_5b_triviaqa"`. What happens to `tau_SC`, and to who counts as "Confident"?